Proyecto: Análisis Exploratorio de Datos del Mercado Automotriz  

Fecha: 2026-05-02_Analisis_Estructura

Autor: (Tu nombre)

Destinatario: Director de Datos / Jefe de Análisis

2. Plan de Exploración Estructurado

import micropip
await micropip.install(['seaborn', 'scipy'], keep_going=True)
print("Librerías instaladas con éxito.")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

# Nombres oficiales en español
nombres_columnas = [
    'riesgo_seguro', 'perdidas_normalizadas', 'marca', 'tipo_combustible', 
    'aspiracion', 'numero_puertas', 'estilo_carroceria', 'traccion', 
    'ubicacion_motor', 'distancia_ejes', 'largo', 'ancho', 'alto', 
    'peso_vacio', 'tipo_motor', 'numero_cilindros', 'tamano_motor', 
    'sistema_combustible', 'diametro_cilindro', 'carrera_piston', 
    'relacion_compresion', 'caballos_fuerza', 'rpm_maximas', 
    'consumo_ciudad_mpg', 'consumo_autopista_mpg', 'precio'
]

# Cargar el archivo sin cabecera y asignarle los nombres
df = pd.read_csv('automoviles.csv', header=None, names=nombres_columnas)

# Verificamos que se cargó bien
print(df.dtypes)

#Veo cuantos datos estoy manejando
print(f"\nDimensiones del dataset: {df.shape}")


#LIMPIEZA DE DATOS
#1. Identificar el tipo de datos de cada columna para entender qué tipo de limpieza es necesaria
# 1. Conversión a numérico (Excelente uso de coerce)
cols_numericas = ['perdidas_normalizadas', 'diametro_cilindro', 'carrera_piston', 
                  'caballos_fuerza', 'rpm_maximas', 'precio']

for col in cols_numericas:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 2. Imputación por la media
# Tip: He añadido 'precio' a la lista de exclusión para no inventar precios
for col in ['perdidas_normalizadas', 'diametro_cilindro', 'carrera_piston', 'caballos_fuerza', 'rpm_maximas']:
    df[col] = df[col].fillna(df[col].mean())

# 3. Imputación por la moda (puertas)
moda_puertas = df['numero_puertas'].value_counts().idxmax()
df['numero_puertas'] = df['numero_puertas'].fillna(moda_puertas)

# 4. Eliminación de filas sin precio
df.dropna(subset=['precio'], axis=0, inplace=True)

# 5. Reinicio el indice
# Al borrar filas con dropna, el índice queda con huecos (ej: 1, 2, 4...). 
# Esto causa errores en visualizaciones y modelos más adelante.
df.reset_index(drop=True, inplace=True)

# 6. Verificacion de los nulos
# Una simple línea para confirmar que no quedaron valores vacíos.
print("--- Conteo de valores nulos finales ---")
print(df.isnull().sum())

# 7. TIPADO CORRECTO 
# Columnas como 'caballos_fuerza' no suelen tener decimales, 
# convertirlas a int ayuda a la legibilidad.
df['caballos_fuerza'] = df['caballos_fuerza'].astype('int')

# --- CONTINUACIÓN DE TU LIMPIEZA ---

# 1. Tu Mapeo de texto a números (Excelente para variables ordinales)
mapeo_cilindros = {'two': 2, 'three': 3, 'four': 4, 'five': 5, 'six': 6, 'eight': 8, 'twelve': 12}
df['numero_cilindros'] = df['numero_cilindros'].map(mapeo_cilindros)

# Ajuste: Ya habíamos imputado la moda en puertas, pero si prefieres 
# mapear a número, este es el momento:
df['numero_puertas'] = df['numero_puertas'].map({'two': 2, 'four': 4})

# 2. Imputación inteligente
# Solo imputamos sobre columnas numéricas para evitar errores de tipo
df.fillna(df.mean(numeric_only=True), inplace=True)

# 3. Creación de Variables Dummy (Para las nominales)
# TIP: Guardamos el original (df) para GRÁFICOS y el nuevo (df_analisis) para MODELOS
df_analisis = pd.get_dummies(df, columns=['tipo_combustible', 'aspiracion', 'estilo_carroceria', 'traccion'])

print("Mapeo y Dummies completados.")
print(f"Columnas en df original: {len(df.columns)}")
print(f"Columnas en df_analisis (con dummies): {len(df_analisis.columns)}")

# 1. Cálculo de correlación lineal (Pearson)
# Especificamos numeric_only=True para procesar solo lo que mapeamos a números
correlaciones = df_analisis.corr(numeric_only=True)['precio'].sort_values(ascending=False)

print("Variables con mayor correlación positiva con el precio:")
print(correlaciones.head(10))

# --- ESTANDARIZACIÓN Y NORMALIZACIÓN ---

# A. Normalización por Simple Feature Scaling (Rango 0 a 1)
# Sobrescribimos las originales o creamos nuevas. 
# Para el print de abajo, usaremos nombres nuevos para claridad.
df['largo_norm'] = df['largo'] / df['largo'].max()
df['ancho_norm'] = df['ancho'] / df['ancho'].max()
df['alto_norm'] = df['alto'] / df['alto'].max()

# B. Estandarización (Z-score)
# Esto centra la media en 0 y la desviación estándar en 1
df['caballos_fuerza_std'] = (df['caballos_fuerza'] - df['caballos_fuerza'].mean()) / df['caballos_fuerza'].std()

print("\n--- Ejemplo de datos normalizados y estandarizados ---")
# Corregido: Ahora las columnas coinciden con lo creado arriba
print(df[['largo_norm', 'caballos_fuerza_std']].head())

# C. Conversión de consumo (MPG a L/100km)
# Es vital para la interpretación de ingeniería automotriz
df['consumo_ciudad_L100km'] = 235 / df['consumo_ciudad_mpg']
df['consumo_autopista_L100km'] = 235 / df['consumo_autopista_mpg']

# D. Verificación de convivencia de unidades
print("\n--- Comparativa de Unidades de Consumo ---")
print(df[['marca', 'consumo_ciudad_mpg', 'consumo_ciudad_L100km', 'precio']].head())

#BINNING
# 1. Definición de bordes (Ya lo tienes)
bins = np.linspace(min(df["caballos_fuerza"]), max(df["caballos_fuerza"]), 4)
nombres_grupos = ['Bajo', 'Medio', 'Alto']

# 2. Aplicamos la segmentación con pd.cut
df['caballos_fuerza_binned'] = pd.cut(df['caballos_fuerza'], bins, labels=nombres_grupos, include_lowest=True)

# 3. Visualización de la segmentación (Fundamental para el análisis)
plt.figure(figsize=(8, 5))
plt.bar(nombres_grupos, df["caballos_fuerza_binned"].value_counts(), color=['skyblue', 'orange', 'salmon'])
plt.title("Distribución de Vehículos por Segmento de Potencia")
plt.xlabel("Categoría de Caballos de Fuerza")
plt.ylabel("Cantidad de Vehículos")
plt.show()

print("Resumen de grupos:")
print(df['caballos_fuerza_binned'].value_counts())


# 1. Definimos los bordes de los grupos para el precio
bins_precio = np.linspace(min(df["precio"]), max(df["precio"]), 4)
nombres_precios = ['Bajo', 'Medio', 'Alto']

# 2. Creamos la columna categórica
df['precio_segmento'] = pd.cut(df['precio'], bins_precio, labels=nombres_precios, include_lowest=True)

# 3. Transformamos a Variables Dummy
dummies_precio = pd.get_dummies(df['precio_segmento'])
dummies_precio = dummies_precio.add_prefix('rango_precio_')

# 4. Unimos al DataFrame
df = pd.concat([df, dummies_precio], axis=1)

print("--- Segmentación de Precios Completada ---")
print(df['precio_segmento'].value_counts())


# Comparación de potencia entre extremos de precio
hp_bajo = df[df['rango_precio_Bajo'] == 1]['caballos_fuerza']
hp_alto = df[df['rango_precio_Alto'] == 1]['caballos_fuerza']

t_stat_p, p_val_p = stats.ttest_ind(hp_bajo, hp_alto)

print(f"\n--- Prueba de Significancia: Potencia vs Segmento de Precio ---")
print(f"Media HP (Precio Bajo): {hp_bajo.mean():.1f}")
print(f"Media HP (Precio Alto): {hp_alto.mean():.1f}")
print(f"Valor p: {p_val_p:.10f}")

# --- AUDITORÍA DE TIPOS PRE-VISUALIZACIÓN ---

# 1. Identificamos columnas de texto (categóricas)
columnas_texto = df.select_dtypes(include=['object']).columns
print("Columnas que aún son texto:", columnas_texto.tolist())

# 2. Identificamos columnas numéricas (para regresión)
columnas_num = df.select_dtypes(include=['number']).columns
print("Columnas listas para análisis numérico:", columnas_num.tolist())

# 1. Creación de dummies para todas las variables categóricas nominales
dummies_combustible = pd.get_dummies(df["tipo_combustible"])
dummies_carroceria = pd.get_dummies(df["estilo_carroceria"])
dummies_traccion = pd.get_dummies(df["traccion"])
dummies_motor_loc = pd.get_dummies(df["ubicacion_motor"])

# 2. Renombrado Descriptivo
# Usamos prefijos para saber exactamente a qué característica pertenece cada columna
dummies_combustible.rename(columns={'gas':'tipo_nafta', 'diesel':'tipo_diesel'}, inplace=True)
dummies_carroceria = dummies_carroceria.add_prefix('carroceria_')
dummies_traccion = dummies_traccion.add_prefix('traccion_')
dummies_motor_loc = dummies_motor_loc.add_prefix('motor_ubicacion_')

# 3. Concatenación al DataFrame principal
# Eliminamos las columnas originales para evitar redundancia en el modelo
df = pd.concat([df, dummies_combustible, dummies_carroceria, dummies_traccion, dummies_motor_loc], axis=1)

# Opcional: Eliminar las columnas originales de texto si ya no las necesitas
# df.drop(['tipo_combustible', 'estilo_carroceria', 'traccion', 'ubicacion_motor'], axis=1, inplace=True)

print("--- Nuevas Variables Dummy Generadas ---")
print(df[['tipo_nafta', 'tipo_diesel', 'carroceria_sedan', 'traccion_fwd', 'motor_ubicacion_front']].head())

# --- AUDITORÍA DE TIPOS PRE-VISUALIZACIÓN ---

# 1. Identificamos columnas de texto (categóricas)
columnas_texto = df.select_dtypes(include=['object']).columns
print("Columnas que aún son texto:", columnas_texto.tolist())

# 2. Identificamos columnas numéricas (para regresión)
columnas_num = df.select_dtypes(include=['number']).columns
print("Columnas listas para análisis numérico:", columnas_num.tolist())

# Definimos el tamaño del gráfico
plt.figure(figsize=(15, 12))

# Calculamos la matriz de correlación solo para valores numéricos
# Esto incluye tus dummies de tracción, combustible, ubicación de motor y rangos de precio
matriz_corr = df.corr(numeric_only=True)

# Creamos el mapa de calor con una paleta divergente (coolwarm)
# annot=False para evitar saturación visual debido a la cantidad de dummies
sns.heatmap(matriz_corr, annot=False, cmap='coolwarm', linewidths=0.3)

plt.title("Mapa de Calor: Correlaciones Globales (Variables Técnicas y Dummies)")
plt.show()

# --- EXTRACCIÓN DE INSIGHTS CLAVE ---
# Vamos a listar las 10 correlaciones más fuertes con el precio para tu reporte
correlaciones_precio = matriz_corr['precio'].sort_values(ascending=False)
print("Top 5 Variables que aumentan el precio:")
print(correlaciones_precio.head(6)) # El 1ero siempre será 'precio'

print("\nTop 5 Variables que disminuyen el precio:")
print(correlaciones_precio.tail(5))

# SECCIÓN 7: PLANTEAMIENTO DE HIPÓTESIS
# 1. Hipótesis de Ingeniería: El tamaño del motor es el predictor principal del precio.
# 2. Hipótesis de Mercado: Los autos con motor trasero pertenecen exclusivamente al segmento de precio "Alto".
# 3. Hipótesis de Eficiencia: A medida que aumenta el precio, aumenta el consumo (L/100km), indicando que el lujo prioriza potencia sobre ahorro.

#1Hipótesis de Arquitectura de Tracción:
#H_0: El tipo de tracción (delantera vs. trasera) no influye en el precio del vehículo.
#H_1: Los vehículos con tracción trasera (rwd) tienen un precio significativamente mayor que los de tracción delantera (fwd).
#2 Hipótesis de Rendimiento Mecánico:H_0: No existe una relación lineal entre los caballos de fuerza y el precio.
#H_1: Existe una correlación positiva fuerte (r > 0.75$) entre la potencia y el valor del vehículo.
#3Hipótesis de Eficiencia y Segmentación:
#H_0: Los vehículos de alto consumo (L/100km) pertenecen equitativamente a todos los rangos de precio.
#H_1: Los vehículos de mayor precio presentan un consumo significativamente más alto, priorizando potencia sobre eficiencia.

#Analsiis visual de correlaciones 
#Hipótesis 1 (Tracción) - Validar Traccion
plt.figure(figsize=(10, 6))
sns.boxplot(x="traccion", y="precio", data=df, palette="Set2")
plt.title("Relación entre Tipo de Tracción y Precio")
plt.xlabel("Tipo de Tracción")
plt.ylabel("Precio (USD)")
plt.show()

#El gráfico muestra que la mediana de precio para la tracción trasera (rwd) es notablemente superior a la delantera (fwd), lo que nos da evidencia inicial para rechazar la H_0.

#Cálculo de Significancia (T-Test)
#Calculamos la probabilidad de que esta diferencia sea producto del azar:
# Filtramos los datos para la prueba
precio_fwd = df[df['traccion_fwd'] == 1]['precio']
precio_rwd = df[df['traccion_rwd'] == 1]['precio']

# Ejecutamos el T-Test para muestras independientes
t_stat, p_val = stats.ttest_ind(precio_fwd, precio_rwd, nan_policy='omit')

print(f"Estadístico T: {t_stat:.4f}")
print(f"Valor p (p-value): {p_val:.10f}")

df.describe()

#Análisis Visual de Correlaciones Clave
#Scatter Plots con una linea de regresion para las 3 variables mas correlacionadas


# Variables a graficar contra el precio
top_vars = ['tamano_motor', 'peso_vacio', 'caballos_fuerza']

plt.figure(figsize=(18, 5))

for i, var in enumerate(top_vars):
    plt.subplot(1, 3, i+1)
    sns.regplot(data=df, x=var, y='precio', scatter_kws={'alpha':0.5}, line_kws={'color':'red'})
    plt.title(f'Relación {var} vs Precio')

plt.tight_layout()
plt.show()

from scipy import stats
# Calculamos Pearson y P-value para Tamaño del Motor vs Precio
coeficiente_pearson, p_valor = stats.pearsonr(df['tamano_motor'], df['precio'])

print("--- Análisis de Correlación Tamaño Motor - Precio ---")
print(f"Coeficiente de Correlación de Pearson: {coeficiente_pearson:.4f}")
print(f"Valor P (P-value): {p_valor:.4e}")

# Interpretación rápida
if p_valor < 0.001:
    print("La relación es estadísticamente significativa.")


#SLR (Regresion Lineal Simple - Tamaño Motor)
#Este modelo asume que el precio depende únicamente de qué tan grande es el motor.
from sklearn.linear_model import LinearRegression

# 1. Definimos las variables
X_simple = df[['tamano_motor']] # Predictor (2D)
Y = df['precio']                # Objetivo

# 2. Creamos y entrenamos el objeto
lm_simple = LinearRegression()
lm_simple.fit(X_simple, Y)

# 3. Calculamos R-cuadrado y Predicciones
r2_simple = lm_simple.score(X_simple, Y)
yhat_simple = lm_simple.predict(X_simple)

print(f"SLR - R-cuadrado (Tamaño Motor): {r2_simple:.4f}")


#MLR (Regresion Lineal Multiple - TAMANO_MOTOR, CABALLOS_FUERZA, PESO_VACIO, CONS_CIUDAD_L100
# 1. Definimos el set de predictores
Z = df[['tamano_motor', 'caballos_fuerza', 'peso_vacio', 'consumo_ciudad_L100km']]

# 2. Entrenamos el modelo múltiple
lm_multiple = LinearRegression()
lm_multiple.fit(Z, Y)

# 3. Calculamos R-cuadrado y Predicciones
r2_multiple = lm_multiple.score(Z, Y)
yhat_multiple = lm_multiple.predict(Z)

print(f"MLR - R-cuadrado (Múltiple: tamano_motor, caballos_fuerza, peso_vacio', consumo_ciudad_L100km): {r2_multiple:.4f}")


#Regresion Polinomica
# 1. Creamos el modelo polinómico (x e y deben ser 1D)
f = np.polyfit(df['tamano_motor'], df['precio'], 3)
p = np.poly1d(f)

# 2. Calculamos R-cuadrado para el polinomio
from sklearn.metrics import r2_score
r2_poly = r2_score(Y, p(df['tamano_motor']))

print(f"Polinomio - R-cuadrado (Grado 3): {r2_poly:.4f}")


#Grafico Polinomio
# 1. Creamos un rango de valores para 'tamano_motor' (de min a max)
# Esto genera 100 puntos intermedios para que la curva sea suave
x_nuevo = np.linspace(df['tamano_motor'].min(), df['tamano_motor'].max(), 100)
y_nuevo = p(x_nuevo)

# 2. Generamos el gráfico
plt.figure(figsize=(10, 6))

# Dibujamos los puntos reales
plt.scatter(df['tamano_motor'], df['precio'], color='steelblue', alpha=0.6, label='Datos Reales')

# Dibujamos la curva del polinomio de grado 3
plt.plot(x_nuevo, y_nuevo, color='firebrick', linewidth=3, label='Regresión Polinómica (Grado 3)')


# Estética del gráfico
plt.title('Ajuste No Lineal: Precio vs Tamaño del Motor', fontsize=14)
plt.xlabel('Tamaño del Motor', fontsize=12)
plt.ylabel('Precio', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7) #Controla la transparencia el alpha, esto es el grillado o tabla le indico que la linea de la tabla sea --, si le coloco un 1 al aplha las lineas son solidas, con negro fuerte 0 transparente

plt.show()



#En motores pequeños (entre 50 y 120), el precio sube de forma lenta y constante
#Sin embargo, a partir de los 150, la pendiente se vuelve mucho más pronunciada. Esto indica que en el segmento de motores grandes, cada unidad adicional de tamaño incrementa el precio de forma exponencial, no proporcional

#Se optó por una regresión polinómica de grado 3 debido a que la relación entre el tamaño del motor y el precio presenta una aceleración no lineal.
#Este modelo captura mejor la dinámica de precios en vehículos de alta cilindrada, donde el valor aumenta de forma exponencial en comparación con los modelos de entrada


from sklearn.metrics import mean_squared_error

mse_simple = mean_squared_error(Y, yhat_simple)
mse_multiple = mean_squared_error(Y, yhat_multiple)

print("\n--- CONCLUSIÓN DEL PROYECTO ---")
print(f"MSE Modelo Simple: {mse_simple:.2f}")
print(f"MSE Modelo Múltiple: {mse_multiple:.2f}")

if r2_multiple > r2_simple and mse_multiple < mse_simple:
    print("El modelo de Regresión Lineal Múltiple es el mejor predictor para este dataset.")


#Gráfico de Distribución Suavizado (kdeplot)
plt.figure(figsize=(10, 6))
# Valores Reales (Rojo)
sns.kdeplot(df['precio'], color="r", label="Precios Reales", fill=True)
# Predicciones (Azul) - Usamos yhat que calculaste en la regresión simple
sns.kdeplot(yhat_simple, color="b", label="Predicción Simple", fill=True)

plt.title('KDE Plot: Comparación de Densidad de Precios')
plt.legend()
plt.show()


plt.figure(figsize=(10, 6))
# Datos Reales
sns.kdeplot(df['precio'], color="r", label="Valores Reales", fill=True)

# Predicciones del modelo Múltiple
sns.kdeplot(yhat_multiple, color="b", label="Predicción Múltiple (MLR)", fill=True)

plt.title('Comparación: Precio Real vs. Predicción del Modelo')
plt.xlabel('Precio (USD)')
plt.ylabel('Densidad')
plt.legend()
plt.show()


#3. El Error Cuadrático Medio (MSE)
from sklearn.metrics import mean_squared_error

mse = mean_squared_error(df['precio'], yhat_simple)
print(f"El Error Cuadrático Medio (MSE) es: {mse:.2f}")
print(f"La raíz del MSE es: {np.sqrt(mse):.2f} USD") 

# Esto último te dice, en promedio, cuántos dólares le erra el modelo

plt.figure(figsize=(10, 6))

# Graficamos los residuos
sns.residplot(x=df['tamano_motor'], y=df['precio'], color='purple')

plt.title('Gráfico de Residuos: tamano_motor vs precio')
plt.xlabel('Tamaño del Motor')
plt.ylabel('Residuos (Error en USD)')
plt.axhline(0, color='black', linestyle='--') # Línea de base cero
plt.show()


# Métricas Simple
mse_s = mean_squared_error(df['precio'], yhat_simple)
r2_s = r2_score(df['precio'], yhat_simple)

# Métricas Múltiple
mse_m = mean_squared_error(df['precio'], yhat_multiple)
r2_m = r2_score(df['precio'], yhat_multiple)

print("--- COMPARATIVA DE MODELOS ---")
print(f"Simple (Motor):  R2: {r2_s:.4f} | MSE: {mse_s:.2f}")
print(f"Múltiple (Z):    R2: {r2_m:.4f} | MSE: {mse_m:.2f}")

# Agregamos la mejor predicción al dataframe
df['prediccion_final'] = yhat_multiple

from sklearn.model_selection import train_test_split
#Evaluación y Refinamiento del Modelo
#train(entrena): set de datos que estudia la relaciones entre el motor y el precio
#Test(prueba):Son datos que el modelo jamás vio. Lo usamos para auditarlo y ver si realmente aprendió a predecir o si simplemente memorizó los datos (lo que llamamos Overfitting)
# 1. Definimos nuestras variables (usando Z para la múltiple que es más potente)
y_data = df['precio']
x_data = df[['tamano_motor', 'caballos_fuerza', 'peso_vacio', 'consumo_ciudad_L100km']]

# 2. Dividimos: 85% entrenamiento, 15% prueba (puedes ajustar el test_size)
x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.15, random_state=1)

print(f"Muestras de entrenamiento: {x_train.shape[0]}")
print(f"Muestras de prueba: {x_test.shape[0]}")

#Entrenamos solo el Set de entrenamiento (train)
#Le pasamos los datos de _train solamente al objeto lm (Regresion Lineal)
lm_final = LinearRegression()
lm_final.fit(x_train, y_train)

# Calculamos el R-cuadrado en ambos sets para comparar
r2_train = lm_final.score(x_train, y_train)
r2_test = lm_final.score(x_test, y_test)

print(f"R-cuadrado en Entrenamiento: {r2_train:.4f}")
print(f"R-cuadrado en Prueba (EL REAL): {r2_test:.4f}")

#Cross-Validation (Validación Cruzada)
#Esto divide el dataset en 4 partes (folds) y entrena/testea el modelo 4 veces rotándolas

from sklearn.model_selection import cross_val_score

# 1. Aplicamos validación cruzada con 4 divisiones (folds)
# Usamos el modelo múltiple (lm_final) y tus datos (x_data, y_data) el cv=4 (es el que indica las divisiones o folds)
# Usamos la funcion  cross_val_score(modelofinal, datos de x, datos y, y las divisiones)
#Esto divide el dataset en 4 partes (folds) y entrena/testea el modelo 4 veces rotándolas
resultados = cross_val_score(lm_final, x_data, y_data, cv=4)

print("--- RESULTADOS DE VALIDACIÓN CRUZADA ---")
print("R-cuadrado por cada división:", resultados)
print(f"Precisión media final: {resultados.mean():.4f}")
print(f"Desviación estándar: {resultados.std():.4f}")

#Regresion de la cresta o Ridge Regression
# Esta técnica ayuda a que el modelo sea más "estable" y no se vea tan afectado por esos valores extremos que están arruinando tu promedio de 0.66
from sklearn.linear_model import Ridge

# 1. Creamos el objeto Ridge con un parámetro alfa (fuerza de regularización)
# Alfa ayuda a que el modelo no sea tan sensible
RidgeModel = Ridge(alpha=0.1)

# 2. Entrenamos
RidgeModel.fit(x_train, y_train)

# 3. Nueva Validación Cruzada para ver si mejoramos la estabilidad
scores_ridge = cross_val_score(RidgeModel, x_data, y_data, cv=4)

print("--- RESULTADOS CON RIDGE REGRESSION ---")
print("Nuevos R-cuadrado:", scores_ridge)
print(f"Nueva Media: {scores_ridge.mean():.4f}")

#Implementación del Árbol de Decisión
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree

#Predecimos el precio pero con la logica de arbol de decision
#Si el valor es cercano a 1 (El peligro del Overfitting)
#Si el valor es cercano a 0 (Underfitting)

# 1. Creamos el modelo. 
# 'max_depth=3' evita que el árbol sea infinito y se vuelva difícil de leer.
dt_model = DecisionTreeRegressor(max_depth=3, random_state=1)

# 2. Entrenamos (usando tus variables x_train y y_train)
dt_model.fit(x_train, y_train)

# 3. Calculamos la precisión con rcuadrado o rquared 
r2_dt = dt_model.score(x_test, y_test)
print(f"R-cuadrado del Árbol de Decisión: {r2_dt:.4f}")

#Visualización Lógica (Minería de Datos)
plt.figure(figsize=(20, 10))
tree.plot_tree(dt_model, 
               feature_names=x_train.columns.tolist(), 
               filled=True, 
               rounded=True, 
               fontsize=10)
plt.title("Mapa de Decisión: Lógica interna del modelo")
plt.show()

#Análisis del Mapa de Decisión
#La Raíz (El primer nodo): El modelo determinó que el tamano_motor es la variable más importante. 
#Si es menor o igual a 182.0, el árbol se va hacia la izquierda (autos más económicos). Si es mayor, se va a la derecha (autos de alta gama).
#Divisiones Estratégicas:
#En el lado de los motores chicos, el siguiente factor decisivo es el peso_vacio.
#En el lado de los motores grandes (derecha), lo que define el precio es el consumo_ciudad_L100km.
#Nodos Finales (Hojas): Los cuadros de abajo te dan el value, que es el precio promedio predicho para ese grupo. 
#Por ejemplo, si un auto cae en la última hoja de la derecha, su valor estimado es $40,786.667.


#Random Forest (El Bosque Aleatorio)
#soluciona los límites que mencionamos antes (evita que un solo árbol se equivoque) creando muchos árboles como el de tu imagen y promediando sus resultados.
from sklearn.ensemble import RandomForestRegressor

# 1. Creamos el Bosque con 100 árboles
# 'n_estimators' es la cantidad de árboles que van a "votar"
randomforest_model = RandomForestRegressor(n_estimators=100, random_state=1)

# 2. Entrenamos
randomforest_model.fit(x_train, y_train)

# 3. Evaluamos la precisión final
r2_rf = randomforest_model.score(x_test, y_test)

print(f"--- RESULTADO FINAL DE MACHINE LEARNING ---")
print(f"R-cuadrado del Random Forest: {r2_rf:.4f}")

#De tener un sistema con un 60% de confiabilidad a uno que tiene un 92.5% de precisión para predecir costos
#"El modelo Random Forest alcanzó un R² de 0.92, demostrando ser el más eficaz para la predicción de precios frente a modelos lineales y de árbol simple".

# Análisis Predictivo de Precios de Automóviles
## Codificación de Variables Categorizadas

# Guardamos siguiendo tu estándar: YYYYMMDD_Analisis_Estructura
import datetime
fecha = datetime.datetime.now().strftime("%Y%m%dd")
nombre_archivo = f"{fecha}_Analisis_Estructura_Final.csv"

df.to_csv(nombre_archivo, index=False)
print(f"Archivo guardado como: {nombre_archivo}")

